<a href="https://colab.research.google.com/github/kouzi5816/ML_Engineering_Repo/blob/main/02_RAGApp/LangChain_Notebook_with_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ライブラリのインストール

In [1]:
!pip install langchain langchain-google-genai langchain-chroma langchain-community pypdf -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 k

ライブラリのインポート

In [2]:
import google.generativeai as genai
from google.colab import userdata
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_4075/970270542.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Googleドライブのマウント

In [3]:
from google.colab import drive
drive.mount('/content/drive')
PDF_path = "/content/drive/MyDrive/RAG/"

Mounted at /content/drive


GeminiのAPI設定と環境確認

In [4]:
# APIキーの取得
# Google ColabのSecretsに保存したAPIキーを読み込み
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

# APIキーの設定
genai.configure(api_key=GEMINI_API_KEY)

# 利用可能なLLMモデルの一覧
print("==========LLMモデル==========")
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

# 利用可能なEmbeddingモデルの一覧
print("==========Embeddingモデル==========")
for m in genai.list_models():
  if 'embedContent' in m.supported_generation_methods:
    print(m.name)

==========LLMモデル==========
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-

LLMのレスポンス確認

In [5]:
# LLMモデルの設定
llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    google_api_key=GEMINI_API_KEY
)

# LLMの導通確認
response = llm.invoke([HumanMessage(content="DX推進とは何ですか？3行で教えてください")])
print(response.content)

DX推進とは、デジタル技術を活用して、企業のビジネスモデルや業務プロセス、組織文化を根本から変革することです。
これにより、顧客体験の向上、新たな価値創造、業務効率化を通じて競争力を高めることを目指します。
単なるIT導入に留まらず、激しい市場変化に対応し、持続的な成長を可能にするための経営戦略です。


プロンプトテンプレートの利用

In [6]:
# LLMモデルの設定
llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    google_api_key=GEMINI_API_KEY
)

# プロンプトのテンプレート設定
prompt = ChatPromptTemplate.from_messages([
    ("system", "あなたは{role}の専門家です。簡潔に答えてください。"),
    ("human", "{question}")
])

# プロンプトテンプレートをLLMに設定
chain = prompt | llm

# プロンプトテンプレートに値を設定しレスポンス確認
response = chain.invoke({
    "role": "レザー製品",
    "question": "海外および日本国内でレザー品質や技術的に優れたブランドを教えてください。"
})
print(response.content)

はい、承知いたしました。海外および日本国内で、レザーの品質と技術的に優れたブランドを簡潔にご紹介します。

---

### 海外ブランド

1.  **エルメス (Hermès)**
    *   最高峰の素材選定、なめし、縫製、コバ処理、全ての工程において卓越した技術と芸術性を誇ります。
2.  **ベルルッティ (Berluti)**
    *   独特のヴェネチアレザーと、職人による手染めのパティーヌ技術が特徴。色彩の深みとエイジングが魅力です。
3.  **ヴァレクストラ (Valextra)**
    *   「イタリアのエルメス」とも称され、ミニマルなデザインと、コバの「コスタ」と呼ばれる塗料処理の美しさが際立ちます。
4.  **ジョンロブ (John Lobb)**
    *   主に靴のブランドですが、最高級の革を厳選し、熟練の職人による堅牢で美しい作りは革小物にも共通します。

### 日本国内ブランド

1.  **GANZO (ガンゾ)**
    *   コードバンやブライドルレザーなど厳選された高級素材と、日本の熟練職人による丁寧な縫製、コバ処理が特徴です。
2.  **WILDSWANS (ワイルドスワンズ)**
    *   堅牢なブライドルレザーなどを多用し、圧倒的な耐久性と、非常に美しい「磨きこまれたコバ」が特徴です。
3.  **土屋鞄製造所**
    *   ランドセルで培われた丁寧な手仕事が魅力。使うほどに馴染む素材選びと、温かみのあるデザインが人気です。
4.  **大峽製鞄 (おおばせいほう)**
    *   皇室御用達として知られ、最高級の素材と、半世紀以上の歴史に裏打ちされた熟練の職人技が光ります。
5.  **万双 (まんそう)**
    *   厳選された素材と熟練の職人技術による高品質な製品を、高いコストパフォーマンスで提供しています。

---

これらのブランドは、いずれも素材への深いこだわりと、それを最大限に活かすための卓越した職人技術によって、世界中で高い評価を得ています。


ベクトルDBへのサンプルデータ格納

In [7]:
# データベースのサンプルデータ
docs = [
    Document(page_content="DX推進には経営層のコミットメントが不可欠である。"),
    Document(page_content="データ基盤の整備がDX成功の前提条件となる。"),
    Document(page_content="機械学習モデルの本番運用にはMLOpsの仕組みが必要だ。"),
    Document(page_content="現場社員のデータリテラシー向上が変革の鍵を握る。"),
    Document(page_content="エネルギー業界では需要予測モデルの活用が進んでいる。"),
]

# Embeddingモデルの設定
embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GEMINI_API_KEY
)

# ベクトルDBに保存（ローカルに永続化）
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding,
    persist_directory="./chroma_db"
)
print("保存完了")

保存完了


ベクトルDBの情報検索

In [8]:
# ベクトルDBから情報取得
results = vectorstore.similarity_search(
    "DXを成功させるために何が必要ですか？",
    k=2  # 上位2件を取得
)

# 検索結果の表示
for i, doc in enumerate(results):
    print(f"--- 検索結果 {i+1} ---")
    print(doc.page_content)

--- 検索結果 1 ---
データ基盤の整備がDX成功の前提条件となる。
--- 検索結果 2 ---
DX推進には経営層のコミットメントが不可欠である。


ベクトルDBへのPDFデータ格納

In [7]:
# PDFを読み込む
# 以下の資料を格納：https://www.meti.go.jp/policy/it_policy/dx/dxreport2.1_gaiyou.pdf
loader = PyPDFLoader('/content/drive/MyDrive/RAG/dxreport2.1_gaiyou.pdf')
pages = loader.load()

# チャンク分割（長文をRAGで扱える長さに切る）
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,    # 1チャンクの最大文字数
    chunk_overlap=50   # チャンク間の重複（文脈を途切れさせない）
)
chunks = splitter.split_documents(pages)
print(f"{len(chunks)}チャンクに分割しました")

# Embeddingモデルの設定
embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GEMINI_API_KEY
)

# ChromaDBに保存
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory="./chroma_db")

28チャンクに分割しました


ベクトルDBの情報検索

In [8]:
# ベクトルDBから情報取得
results = vectorstore.similarity_search(
    "デジタル産業と既存産業の違いはなんですか？",
    k=2  # 上位2件を取得
)

# 検索結果の表示
for i, doc in enumerate(results):
    print(f"--- 検索結果 {i+1} ---")
    print(doc.page_content)

--- 検索結果 1 ---
11
3.3 デジタル産業と既存産業の比較
環境の変化
①
顧客体験
の向上
②
市場変化
への迅速
な対応
 インターネット、スマートフォン
の普及により顧客接点がデ
ジタル化
①顧客体験の向上が主戦場に
②市場の変化が加速し、
迅速な対応が必要に
 作らずに使う、組み合わせる
 小さくつくり、迅速にスケール
 クラウド技術、アジャイルでの
内製、DevOpsを活用
 顧客価値にフォーカスした価
値創出のネットワーク
 データを活用し、顧客一人
ひとりを深く理解
デジタル産業 既存の産業
(例：ITベンダ産業)
顧客 消費者・個人 発注者
チャネル オンライン/デジタルサービス オフライン
価値の源泉 顧客とのインタラクションとコラボレーション 労働力
キーアクティビティ 課題の解決・顧客体験の向上 要件の実現
スピード リアルタイム バッチ
何を提供するか 価値 労働量
商流 価値を中心としたつながり 固定的な取引関係
収益の流れ 価値の受け取り手→創出者 元請け→下請け
産業構造 ネットワーク型 ピラミッド型
選定基準 ビジョン・共感 調達コスト・労働分配
--- 検索結果 2 ---
12
3.4 デジタル産業の構造と企業類型（1／2）
 デジタル産業は、ソフトウェアやインターネットにより、グローバルにスケール可能で労働
量によらない特性にあり、資本の大小や中央・地方の別なく、価値創出に参画できる。
 市場との対話の中で迅速に変化する必要性や、1社で対応できない多様な価値を結び
つける必要性から、固定的ではないネットワーク型の構造となる。
業界新ビジネス・サービスの
提供の主体
P/F
 P/F
 P/F
P/F
機能
（Value Chain）
共通プラットフォームの
提供主体
A業界 B業界
大企業
中小零細
DXに必要な技術を
提供するパートナー
デ
ジ
タ
ル
産
業
の
業
界
構
造
既
存
産
業
の
業
界
構
造
大企業
中小零細企業
新興ベンチャー
凡例
2
3
4
DX
企業の変革を共に
推進するパートナー １ 伴走 伴走


プロンプトテンプレートの利用

In [11]:
# LLMモデルの設定
llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    google_api_key=GEMINI_API_KEY
)

# プロンプトテンプレートの設定
prompt = ChatPromptTemplate.from_template("""
以下のコンテキストのみを使って質問に答えてください。
コンテキストに答えがない場合は「情報が見つかりませんでした」と答えてください。

コンテキスト:
{context}

質問: {question}
""")

# Retriever：vectorstoreの検索機能をLangChain形式に変換
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# RAGチェーンの組み立て（パイプでつなぐだけ）
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 実行
question = "デジタル産業と既存産業の違いを教えてください？"
answer = rag_chain.invoke(question)
print(answer)

デジタル産業と既存産業の主な違いは以下の通りです。

| 項目         | デジタル産業                   | 既存の産業 (例：ITベンダ産業) |
| :----------- | :----------------------------- | :----------------------------- |
| 顧客         | 消費者・個人                   | 発注者                         |
| チャネル     | オンライン/デジタルサービス    | オフライン                     |
| 価値の源泉   | 顧客とのインタラクションとコラボレーション | 労働力                         |
| キーアクティビティ | 課題の解決・顧客体験の向上   | 要件の実現                     |
| スピード     | リアルタイム                   | バッチ                         |
| 何を提供するか | 価値                           | 労働量                         |
| 商流         | 価値を中心としたつながり       | 固定的な取引関係               |
| 収益の流れ   | 価値の受け取り手→創出者       | 元請け→下請け                 |
| 産業構造     | ネットワーク型                 | ピラミッド型                   |
| 選定基準     | ビジョン・共感                 | 調達コスト・労働分配           |

また、デジタル産業は、インターネット、スマートフォンの普及により顧客接点がデジタル化し、顧客体験の向上が主戦場となり、市場の変化が加速し迅速な対応が必要とされています。これにより、作らずに使う、組み合わせる、小さくつくり迅速にスケールするといった特徴があります。クラウド技術、アジャイルでの内製、DevOpsを活用し、顧客価値にフォーカスした価値創出のネ

In [12]:
# 検索結果と回答を同時に取得
retrieved_docs = retriever.invoke(question)

print("=== 参照した箇所 ===")
for i, doc in enumerate(retrieved_docs):
    print(f"[{i+1}] {doc.page_content[:100]}...")

print("\n=== 回答 ===")
print(rag_chain.invoke(question))

=== 参照した箇所 ===
[1] 11
3.3 デジタル産業と既存産業の比較
環境の変化
①
顧客体験
の向上
②
市場変化
への迅速
な対応
 インターネット、スマートフォン
の普及により顧客接点がデ
ジタル化
①顧客体験の向上が...
[2] 12
3.4 デジタル産業の構造と企業類型（1／2）
 デジタル産業は、ソフトウェアやインターネットにより、グローバルにスケール可能で労働
量によらない特性にあり、資本の大小や中央・地方の別なく、価...
[3] 5
2.1 ユーザー企業とベンダー企業の相互依存関係
 既存産業の業界構造は、ユーザー企業は委託による「コストの削減」を、ベンダー企業
は受託による「低リスク・長期安定ビジネスの享受」というWin-...

=== 回答 ===
デジタル産業と既存産業の主な違いは以下の通りです。

| 項目           | デジタル産業                   | 既存の産業（例：ITベンダ産業） |
| :------------- | :----------------------------- | :----------------------------- |
| **顧客**       | 消費者・個人                   | 発注者                         |
| **チャネル**   | オンライン/デジタルサービス    | オフライン                     |
| **価値の源泉** | 顧客とのインタラクションとコラボレーション | 労働力                         |
| **キーアクティビティ** | 課題の解決・顧客体験の向上   | 要件の実現                     |
| **スピード**   | リアルタイム                   | バッチ                         |
| **提供するもの** | 価値                           | 労働量                         |
| **商流**       | 価値を中心としたつながり       | 固定的な取引関係           